# 03 — Confidence Calibration via Temperature Scaling

This notebook applies post-hoc temperature scaling (Guo et al., 2017) to the trained EfficientNet-B3 detector from notebook 02.

**What temperature scaling does:** Modern neural networks tend to be overconfident — a model reporting 95% confidence may only be correct 80% of the time. Temperature scaling learns a single parameter *T* that softens the model's output distribution:

```
calibrated_probs = softmax(logits / T)
```

When *T* > 1, predictions become less extreme (closer to uniform), reducing overconfidence.

**This notebook:**
1. Loads the trained checkpoint and validation data
2. Collects raw logits on the validation set
3. Measures calibration **before** temperature scaling (ECE + reliability diagram)
4. Learns the optimal temperature *T* on the validation set
5. Measures calibration **after** temperature scaling (ECE + reliability diagram)
6. Produces a side-by-side comparison plot
7. Saves the results to `baseline_results.json`

**Prerequisites:** Run `02_baseline_training.ipynb` first to produce a trained checkpoint.

In [ ]:
# Cell 1 — Imports, paths, device, load checkpoint
import os, sys, json
import numpy as np
import torch
import torch.nn as nn

REPO_DIR = "/content/ai-image-detection"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

DRIVE_ROOT = "/content/drive/MyDrive/ai-image-detection"
DATA_DIR = os.path.join(REPO_DIR, "data", "raw", "cifake")
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
PLOTS_DIR = os.path.join(REPO_DIR, "outputs", "plots")
RESULTS_DIR = os.path.join(REPO_DIR, "outputs", "results")
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

from src.model.detector import build_detector, load_checkpoint
from src.utils.data_loader import get_cifake_loaders

model = build_detector(pretrained=False).to(device)
checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_detector.pth")
epoch_loaded, ckpt_metrics = load_checkpoint(model, checkpoint_path)
model.to(device)
model.eval()

print(f"Loaded checkpoint from epoch {epoch_loaded}")
print(f"  val_acc: {ckpt_metrics['val_acc']:.4f}")

_, val_loader, test_loader = get_cifake_loaders(DATA_DIR, batch_size=32, num_workers=2)

In [ ]:
# Cell 2 — Collect raw logits on the validation set
from src.evaluation.calibration import collect_logits

val_logits, val_labels = collect_logits(model, val_loader, device)

print(f"Collected logits: {val_logits.shape}")
print(f"Labels:           {val_labels.shape}")
print(f"Class distribution: {dict(zip(*np.unique(val_labels.numpy(), return_counts=True)))}")

## Pre-Calibration Analysis

Before applying temperature scaling, we measure how well the model's raw confidence scores match its actual accuracy. A perfectly calibrated model sits on the diagonal of the reliability diagram.

In [ ]:
# Cell 3 — Pre-calibration: ECE and reliability diagram
from src.evaluation.metrics import compute_ece, plot_reliability_diagram

probs_before = torch.softmax(val_logits, dim=1).numpy()
ece_before = compute_ece(probs_before, val_labels.numpy())

print(f"ECE (before temperature scaling): {ece_before:.4f}")

plot_reliability_diagram(
    probs_before,
    val_labels.numpy(),
    os.path.join(PLOTS_DIR, "reliability_pre_calibration.png"),
)
print(f"Saved: {PLOTS_DIR}/reliability_pre_calibration.png")

## Learn Optimal Temperature

We optimise a single scalar *T* on the validation set using L-BFGS to minimise the negative log-likelihood. This is the method from Guo et al. (2017) — despite having only one parameter, it matches or outperforms more complex calibration methods.

In [ ]:
# Cell 4 — Learn temperature on validation set
from src.evaluation.calibration import find_temperature

learned_temperature = find_temperature(model, val_loader, device)

print(f"\nLearned temperature: {learned_temperature:.4f}")
if learned_temperature > 1.0:
    print("T > 1 indicates the model was overconfident (as expected for modern DNNs).")
else:
    print("T <= 1 indicates the model was underconfident.")

## Post-Calibration Analysis

Now we apply the learned temperature to the same validation logits and re-measure ECE. If calibration worked, the bars in the reliability diagram should move closer to the diagonal and ECE should decrease.

In [ ]:
# Cell 5 — Post-calibration: ECE and reliability diagram
from src.evaluation.calibration import apply_temperature

probs_after = apply_temperature(val_logits.numpy(), learned_temperature)
ece_after = compute_ece(probs_after, val_labels.numpy())

print(f"ECE (after temperature scaling):  {ece_after:.4f}")
print(f"ECE reduction:                    {ece_before - ece_after:.4f}")

plot_reliability_diagram(
    probs_after,
    val_labels.numpy(),
    os.path.join(PLOTS_DIR, "reliability_post_calibration.png"),
)
print(f"Saved: {PLOTS_DIR}/reliability_post_calibration.png")

## Side-by-Side Comparison

A combined figure showing the reliability diagram before and after calibration. The dotted diagonal represents perfect calibration.

In [ ]:
# Cell 6 — Side-by-side reliability diagrams
import matplotlib.pyplot as plt

n_bins = 10
bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
bin_centres = [(bin_boundaries[i] + bin_boundaries[i + 1]) / 2 for i in range(n_bins)]
width = 1.0 / n_bins


def _bin_accuracies(probs, labels):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels).astype(float)
    accs = []
    for i in range(n_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        accs.append(float(accuracies[mask].mean()) if mask.sum() > 0 else 0.0)
    return accs


labels_np = val_labels.numpy()
accs_before = _bin_accuracies(probs_before, labels_np)
accs_after = _bin_accuracies(probs_after, labels_np)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.bar(bin_centres, accs_before, width=width * 0.9, alpha=0.7, label="Accuracy")
ax1.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax1.set_xlabel("Confidence")
ax1.set_ylabel("Accuracy")
ax1.set_title(f"Before (ECE = {ece_before:.4f})")
ax1.legend(loc="upper left")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)

ax2.bar(bin_centres, accs_after, width=width * 0.9, alpha=0.7, color="green", label="Accuracy")
ax2.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax2.set_xlabel("Confidence")
ax2.set_ylabel("Accuracy")
ax2.set_title(f"After  (ECE = {ece_after:.4f}, T = {learned_temperature:.2f})")
ax2.legend(loc="upper left")
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

fig.suptitle("Reliability Diagrams: Before vs After Temperature Scaling", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "calibration_comparison.png"), dpi=150)
plt.show()
print(f"Saved: {PLOTS_DIR}/calibration_comparison.png")

## Test Set Evaluation + Save Results

Finally, we evaluate the calibrated model on the **test set** and save all metrics to a JSON file. This file will be used in Phase 2 as the baseline for comparison against generalisation results on other generators.

In [ ]:
# Cell 7 — Test set evaluation with calibration + save results
from src.evaluation.metrics import compute_accuracy, compute_auc
from tqdm.auto import tqdm

test_logits, test_labels = collect_logits(model, test_loader, device)

test_probs_before = torch.softmax(test_logits, dim=1).numpy()
test_probs_after = apply_temperature(test_logits.numpy(), learned_temperature)

test_preds = np.argmax(test_probs_before, axis=1)
test_labels_np = test_labels.numpy()

test_acc = compute_accuracy(test_preds, test_labels_np)
test_auc = compute_auc(test_probs_before, test_labels_np)
test_ece_before = compute_ece(test_probs_before, test_labels_np)
test_ece_after = compute_ece(test_probs_after, test_labels_np)

print(f"{'='*45}")
print(f"  Test Results (CIFAKE) — Calibrated")
print(f"{'='*45}")
print(f"  Accuracy:              {test_acc:.4f}")
print(f"  AUC-ROC:               {test_auc:.4f}")
print(f"  ECE (before scaling):  {test_ece_before:.4f}")
print(f"  ECE (after scaling):   {test_ece_after:.4f}")
print(f"  Temperature:           {learned_temperature:.4f}")
print(f"{'='*45}")

# Save structured results for Phase 2 comparison
results = {
    "dataset": "CIFAKE",
    "model": "EfficientNet-B3",
    "checkpoint_epoch": int(epoch_loaded),
    "test_accuracy": round(test_acc, 4),
    "test_auc": round(test_auc, 4),
    "ece_before_calibration": round(test_ece_before, 4),
    "ece_after_calibration": round(test_ece_after, 4),
    "temperature": round(learned_temperature, 4),
    "checkpoint_path": "outputs/checkpoints/best_detector.pth",
}

results_path = os.path.join(RESULTS_DIR, "baseline_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {results_path}")